# 03 — Clustering: Segmentasi Pelanggan (K-Means)

Notebook ini mengelompokkan pelanggan ke dalam klaster berdasarkan usia, jumlah order, dan total pengeluaran menggunakan K-Means dari Spark MLlib.

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.insert(0, "/home/jovyan/work")

from analysis.spark_session import create_spark_session
spark = create_spark_session("03 - Clustering")

## 2. Baca Fitur Pelanggan dari Processed Zone

In [ ]:
BUCKET = "datalake"
df_features = spark.read.csv(f"s3a://{BUCKET}/processed/customer_features/", header=True, inferSchema=True)
df_features.show(5)
print(f"Total baris: {df_features.count()}")

## 3. Jalankan K-Means (k=3)

In [ ]:
from analysis.mining.clustering import run_kmeans

df_clustered, model = run_kmeans(
    df_features,
    feature_cols=["age", "total_orders", "total_spend"],
    k=3,
)

df_clustered.select("customer_name", "city", "age", "total_orders", "total_spend", "cluster").show()

## 4. Ringkasan per Klaster

In [ ]:
from pyspark.sql import functions as F

df_clustered.groupBy("cluster").agg(
    F.count("customer_id").alias("jumlah_pelanggan"),
    F.round(F.avg("age"), 1).alias("rata_usia"),
    F.round(F.avg("total_orders"), 1).alias("rata_orders"),
    F.round(F.avg("total_spend"), 0).alias("rata_spend"),
).orderBy("cluster").show()

## 5. Visualisasi Klaster

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pdf = df_clustered.toPandas()
colors = {0: "steelblue", 1: "seagreen", 2: "darkorange"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cluster_id, group in pdf.groupby("cluster"):
    axes[0].scatter(group["age"], group["total_spend"], label=f"Klaster {cluster_id}", color=colors[cluster_id], alpha=0.7)
axes[0].set_xlabel("Usia")
axes[0].set_ylabel("Total Spend")
axes[0].set_title("Klaster: Usia vs Total Spend")
axes[0].legend()

for cluster_id, group in pdf.groupby("cluster"):
    axes[1].scatter(group["total_orders"], group["total_spend"], label=f"Klaster {cluster_id}", color=colors[cluster_id], alpha=0.7)
axes[1].set_xlabel("Total Orders")
axes[1].set_ylabel("Total Spend")
axes[1].set_title("Klaster: Orders vs Total Spend")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Centroid Klaster

In [ ]:
# Ambil centroid dari tahap KMeans dalam pipeline
kmeans_model = model.stages[-1]
for i, center in enumerate(kmeans_model.clusterCenters()):
    print(f"Klaster {i}: {center}")

## 7. Simpan Hasil ke MinIO

In [ ]:
df_clustered.write.mode("overwrite").option("header", True) \n    .csv(f"s3a://{BUCKET}/processed/customer_clusters/")
print("Hasil clustering disimpan ke processed zone.")